In [1]:
# -------------------------------------------------------
# Clasificación de géneros musicales con el mejor modelo
# Dataset: GTZAN (features_3_sec.csv)
# -------------------------------------------------------

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow import keras

# 1. Fijar semilla para reproducibilidad
seed = 12
np.random.seed(seed)
tf.random.set_seed(seed)

# 2. Cargar dataset (debe estar en la misma carpeta)
df = pd.read_csv('features_3_sec.csv')
print("Dataset:", df.shape)

# 3. Eliminar columnas que no se usan
df.drop(['filename', 'length'], axis=1, inplace=True)

# 4. Separar features y etiquetas
y = df.pop('label')      # etiquetas (género)
X = df                   # features numéricos

# 5. Codificar las etiquetas a números
# el CSV de Kaggle ya suele traer los labels como texto (blues, classical, etc.)
# aquí las convertimos a índice
labels_unicos = y.unique()
label_to_index = {label: idx for idx, label in enumerate(labels_unicos)}
index_to_label = {idx: label for label, idx in label_to_index.items()}
y_num = y.map(label_to_index)

# 6. Partir en train, dev y test (70/20/10)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y_num,
    train_size=0.7,
    random_state=seed,
    stratify=y_num
)

X_dev, X_test, y_dev, y_test = train_test_split(
    X_temp, y_temp,
    train_size=0.66,     # 0.66 de 0.30 ≈ 0.20 total
    random_state=seed,
    stratify=y_temp
)

print(f"Train: {X_train.shape[0]} | Dev: {X_dev.shape[0]} | Test: {X_test.shape[0]}")

# 7. Escalar datos
scaler = StandardScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_dev   = pd.DataFrame(scaler.transform(X_dev),   columns=X_train.columns)
X_test  = pd.DataFrame(scaler.transform(X_test),  columns=X_train.columns)

# 8. Definir el MEJOR modelo (multicapa con dropout)
def build_best_model(input_dim, num_classes):
    model = keras.models.Sequential([
        keras.layers.Dense(512, activation='relu', input_shape=(input_dim,)),
        keras.layers.Dropout(0.2),

        keras.layers.Dense(256, activation='relu'),
        keras.layers.Dropout(0.2),

        keras.layers.Dense(128, activation='relu'),
        keras.layers.Dropout(0.2),

        keras.layers.Dense(64, activation='relu'),
        keras.layers.Dropout(0.2),

        keras.layers.Dense(num_classes, activation='softmax'),
    ])
    return model

num_features = X_train.shape[1]
num_classes = len(labels_unicos)

model = build_best_model(num_features, num_classes)
model.summary()

# 9. Compilar
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# 10. Entrenar
EPOCHS = 20
BATCH_SIZE = 128

history = model.fit(
    X_train, y_train,
    validation_data=(X_dev, y_dev),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1
)

# 11. Evaluar en test
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"\nAccuracy en test: {test_acc:.4f}")

# 12. Función para predecir el género de un vector de features ya extraído
def predecir_genero(feature_row):
    """
    feature_row: pandas Series o array con las mismas columnas que X
    """
    if isinstance(feature_row, pd.Series):
        feature_row = feature_row.values.reshape(1, -1)
    else:
        feature_row = np.array(feature_row).reshape(1, -1)

    feature_row = scaler.transform(feature_row)
    probs = model.predict(feature_row)
    pred_idx = np.argmax(probs, axis=1)[0]
    return index_to_label[pred_idx]

# Ejemplo de predicción con un registro del test
ejemplo = X_test.iloc[0]
real = index_to_label[y_test.iloc[0]]
pred = predecir_genero(ejemplo)
print(f"\nGénero real: {real} | Género predicho: {pred}")


Dataset: (9990, 60)
Train: 6993 | Dev: 1978 | Test: 1019


C:\ProgramData\miniconda3\Lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (Dense)                        │ (None, 512)                 │          29,696 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 512)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 256)                 │         131,328 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 128)                 │          32,896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 64)                  │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_3 (Dropout)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_4 (Dense)                      │ (None, 10)                  │             650 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 202,826 (792.29 KB)

 Trainable params: 202,826 (792.29 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
55/55 ━━━━━━━━━━━━━━━━━━━━ 4s 17ms/step - accuracy: 0.4010 - loss: 1.6855 - val_accuracy: 0.6047 - val_loss: 1.1664
Epoch 2/20
55/55 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.5946 - loss: 1.1546 - val_accuracy: 0.7113 - val_loss: 0.8824
Epoch 3/20
55/55 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.6784 - loss: 0.9414 - val_accuracy: 0.7381 - val_loss: 0.7610
Epoch 4/20
55/55 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.7274 - loss: 0.8061 - val_accuracy: 0.7624 - val_loss: 0.7095
Epoch 5/20
55/55 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.7582 - loss: 0.7259 - val_accuracy: 0.7836 - val_loss: 0.6437
Epoch 6/20
55/55 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.7865 - loss: 0.6383 - val_accuracy: 0.7983 - val_loss: 0.6049
Epoch 7/20
55/55 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.8049 - loss: 0.5703 - val_accuracy: 0.8124 - val_loss: 0.5667
Epoch 8/20
55/55 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.8283 - loss: 0.5168 - val_accuracy: 0.8301 - v

C:\ProgramData\miniconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 347ms/step

Género real: metal | Género predicho: rock
